# Análisis geoespacial para retail: Tienda Urbana

Este notebook es un caso introductorio para aprender geoespacial con una situación de retail.
**Tienda Urbana** quiere identificar cuál de tres zonas de clientes está más cerca de su tienda principal
y qué zonas quedan dentro de un radio de atención de 8 km.

Los datos son inventados y sirven para aprender. La distancia es en línea recta; no representa la ruta
real de un cliente. El notebook utiliza GeoPandas para trabajar con puntos, mapas y distancias.

**Cómo ejecutarlo en Colab:** súbalo desde **Archivo → Subir notebook** y seleccione **Entorno de ejecución → Ejecutar todas**.

## 1. Preparar las librerías

Esta celda instala GeoPandas y Matplotlib si aún no están disponibles. GeoPandas añade una columna
especial llamada `geometry` a las tablas y permite calcular relaciones entre lugares. En Colab,
`%pip` instala en el mismo entorno que utiliza el notebook.

In [ ]:
%pip install -q "geopandas==1.1.4" "matplotlib>=3.8,<4" "pandas>=2.2,<3" "numpy>=1.26,<3"

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from pathlib import Path

SALIDA = Path("salidas_retail")
SALIDA.mkdir(exist_ok=True)
pd.options.display.float_format = "{:,.2f}".format

## 2. El problema de negocio

Una tabla de ventas puede decir cuántos clientes tiene una zona, pero no muestra fácilmente qué tan
cerca está esa zona de la tienda. Con geoespacial podemos colocar los lugares en un mapa y ordenar
las zonas por distancia.

En este ejemplo usamos tres zonas representativas. La columna `clientes_mes` es demanda mensual
simulada; no son personas identificadas. La pregunta es: **¿qué zonas están dentro de 8 km y cuál es la más cercana?**

In [ ]:
datos = pd.DataFrame({
    "lugar": ["Tienda principal", "Zona Norte", "Zona Este", "Zona Oeste"],
    "tipo": ["Tienda", "Cliente", "Cliente", "Cliente"],
    "longitud": [-100.30, -100.25, -100.18, -100.42],
    "latitud": [25.67, 25.70, 25.73, 25.65],
    "clientes_mes": [np.nan, 420, 310, 180],
})
display(datos)

La primera fila es la tienda; las otras tres son zonas de clientes. Las coordenadas están en grados.
Todavía no debemos restarlas y llamarlas kilómetros: primero necesitamos un sistema de coordenadas métrico.

## 3. Dibujar los lugares

`points_from_xy` convierte longitud y latitud en puntos. `EPSG:4326` indica que son coordenadas
geográficas. El mapa responde una pregunta visual: **¿dónde están la tienda y sus zonas?**

In [ ]:
lugares = gpd.GeoDataFrame(
    datos.copy(),
    geometry=gpd.points_from_xy(datos["longitud"], datos["latitud"]),
    crs="EPSG:4326",
)
fig, ax = plt.subplots(figsize=(8, 5))
lugares[lugares["tipo"] == "Cliente"].plot(ax=ax, color="#e39b32", markersize=110, label="Zonas de clientes")
lugares[lugares["tipo"] == "Tienda"].plot(ax=ax, color="#246a95", marker="*", markersize=260, label="Tienda")
for fila in lugares.itertuples():
    ax.annotate(fila.lugar, (fila.geometry.x, fila.geometry.y), xytext=(5, 6), textcoords="offset points")
ax.set(title="Tienda Urbana y sus zonas de clientes", xlabel="Longitud", ylabel="Latitud")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(SALIDA / "01_mapa_retail.png", bbox_inches="tight")
plt.show()

Los puntos naranjas representan zonas de clientes y la estrella azul representa la tienda. La posición
ayuda a orientarnos, pero aún necesitamos una medida para ordenar las zonas.

## 4. Medir distancias en kilómetros

Transformamos los puntos a `EPSG:32614`, una proyección adecuada para esta zona. En ella las unidades
son metros. Después calculamos la distancia de cada zona a la tienda y dividimos entre 1,000 para obtener
kilómetros. Esta transformación es importante: medir directamente en grados produciría unidades difíciles de interpretar.

In [ ]:
lugares_m = lugares.to_crs("EPSG:32614")
tienda = lugares_m.loc[lugares_m["tipo"] == "Tienda", "geometry"].iloc[0]
zonas = lugares_m[lugares_m["tipo"] == "Cliente"].copy()
zonas["distancia_km"] = zonas.geometry.distance(tienda) / 1000
zonas["dentro_radio_8km"] = zonas["distancia_km"] <= 8
resultado = zonas[["lugar", "clientes_mes", "distancia_km", "dentro_radio_8km"]].sort_values("distancia_km")
display(resultado)

In [ ]:
mas_cercana = resultado.iloc[0]
clientes_cubiertos = resultado.loc[resultado["dentro_radio_8km"], "clientes_mes"].sum()
clientes_totales = resultado["clientes_mes"].sum()
porcentaje_cubierto = 100 * clientes_cubiertos / clientes_totales
display(Markdown(f"""
**Resultado:** la zona más cercana es **{mas_cercana['lugar']}**, a **{mas_cercana['distancia_km']:.1f} km** en línea recta.
Dentro de 8 km quedan **{int(resultado['dentro_radio_8km'].sum())} de 3 zonas** y **{int(clientes_cubiertos)} de {int(clientes_totales)} clientes mensuales simulados ({porcentaje_cubierto:.1f}%)**.
"""))

La tabla ordenada permite comparar las zonas. El porcentaje de clientes cubiertos pesa las zonas por
su demanda: una zona con muchos clientes puede ser más importante que varias zonas pequeñas.
La distancia sigue siendo geométrica; para saber cuánto tarda un cliente necesitaríamos calles y tráfico.

## 5. Ver el radio de atención en el mapa

Un `buffer` dibuja un círculo de 8 km alrededor de la tienda. Lo usamos para comunicar visualmente
la regla de cobertura. La clasificación oficial del ejercicio es la columna `dentro_radio_8km`,
calculada con la distancia numérica.

In [ ]:
radio = gpd.GeoDataFrame(
    {"nombre": ["Radio de atención"]},
    geometry=[tienda.buffer(8_000)],
    crs="EPSG:32614",
).to_crs("EPSG:4326")
lugares_mapa = lugares.to_crs("EPSG:4326")
fig, ax = plt.subplots(figsize=(8, 5))
radio.plot(ax=ax, color="#8ed1c5", alpha=0.35, edgecolor="#218c74", label="Radio 8 km")
lugares_mapa[lugares_mapa["tipo"] == "Cliente"].plot(ax=ax, color="#e39b32", markersize=110, label="Zonas de clientes")
lugares_mapa[lugares_mapa["tipo"] == "Tienda"].plot(ax=ax, color="#246a95", marker="*", markersize=260, label="Tienda")
for fila in lugares_mapa.itertuples():
    ax.annotate(fila.lugar, (fila.geometry.x, fila.geometry.y), xytext=(5, 6), textcoords="offset points")
ax.set(title="Cobertura geométrica de Tienda Urbana", xlabel="Longitud", ylabel="Latitud")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(SALIDA / "02_cobertura_retail.png", bbox_inches="tight")
plt.show()

La zona verde es un radio de referencia, no un mapa de calles. Una zona fuera del círculo no está
automáticamente perdida: podría tener clientes valiosos o una ruta rápida. La figura sirve para decidir
qué conviene investigar después.

## 6. Conclusiones del caso retail

Tienda Urbana utilizó una tabla muy pequeña para responder una pregunta concreta. El método fue:
crear puntos, dibujarlos, transformar el sistema de coordenadas, medir distancias y aplicar una regla de cobertura.

En un proyecto real, el siguiente paso sería incorporar ventas reales por zona, tiempo de viaje, calles,
competidores, densidad de población y horarios. La distancia en línea recta es una primera aproximación,
no una decisión definitiva de apertura o cierre de una tienda.

In [ ]:
mas_cercana = resultado.iloc[0]
clientes_cubiertos = resultado.loc[resultado["dentro_radio_8km"], "clientes_mes"].sum()
clientes_totales = resultado["clientes_mes"].sum()
porcentaje_cubierto = 100 * clientes_cubiertos / clientes_totales

resumen = {
    "tienda": "Tienda Urbana",
    "zona_mas_cercana": mas_cercana["lugar"],
    "distancia_mas_cercana_km": float(mas_cercana["distancia_km"]),
    "zonas_dentro_8km": int(resultado["dentro_radio_8km"].sum()),
    "clientes_cubiertos": int(clientes_cubiertos),
    "clientes_totales": int(clientes_totales),
    "cobertura_clientes_pct": float(porcentaje_cubierto),
}
pd.DataFrame([resumen]).to_csv(SALIDA / "resumen_tienda_urbana.csv", index=False)
resultado.to_csv(SALIDA / "distancias_zonas_clientes.csv", index=False)
lugares.to_crs("EPSG:4326").to_file(SALIDA / "lugares_retail.geojson", driver="GeoJSON")
print("Archivos guardados en:", SALIDA.resolve())
display(pd.DataFrame([resumen]))